# Graph Theory for Visual Learners

In [ ]:
import math
import time
from dataclasses import dataclass, field
from typing import Any, Callable, Iterable, Mapping


@dataclass
class Canvas:
    """Global canvas styling shared by notebook renderers."""

    background_color: str = "#ffffff"


@dataclass
class Camera:
    """2D camera state with smooth interpolation toward target centers."""

    center: tuple[float, float, float] = (0.0, 0.0, 0.0)
    height: float = 480.0
    _start_center: tuple[float, float, float] = field(default=(0.0, 0.0, 0.0), init=False, repr=False)
    _target_center: tuple[float, float, float] = field(default=(0.0, 0.0, 0.0), init=False, repr=False)
    _start_height: float = field(default=480.0, init=False, repr=False)
    _target_height: float = field(default=480.0, init=False, repr=False)
    _start_time: float = field(default=0.0, init=False, repr=False)
    _duration: float = field(default=0.0, init=False, repr=False)
    _windup: float = field(default=0.0, init=False, repr=False)
    _winddown: float = field(default=0.0, init=False, repr=False)
    _moving: bool = field(default=False, init=False, repr=False)

    def __post_init__(self):
        """Validate constructor inputs and align interpolation anchors with initial camera state."""
        self.center = self._coerce_center(self.center, name="center")
        self.height = self._coerce_finite_float(self.height, name="height")
        self._start_center = self.center
        self._target_center = self.center
        self._start_height = self.height
        self._target_height = self.height

    @staticmethod
    def _coerce_finite_float(value, *, name: str) -> float:
        """Coerce values to finite floats for predictable camera interpolation math."""
        try:
            numeric = float(value)
        except Exception as exc:
            raise ValueError(f"{name} must be a finite number") from exc
        if not math.isfinite(numeric):
            raise ValueError(f"{name} must be a finite number")
        return numeric

    @classmethod
    def _coerce_center(cls, raw_center: Iterable[Any], *, name: str) -> tuple[float, float, float]:
        """Normalize center-like constructor input into a strict `(x, y, z)` float tuple."""
        if raw_center is None:
            raise ValueError(f"{name} cannot be None")
        try:
            values = list(raw_center)
        except TypeError as exc:
            raise ValueError(f"{name} must be an iterable with exactly 3 values") from exc
        if len(values) != 3:
            raise ValueError(f"{name} must contain exactly 3 values")
        return (
            cls._coerce_finite_float(values[0], name=f"{name}[0]"),
            cls._coerce_finite_float(values[1], name=f"{name}[1]"),
            cls._coerce_finite_float(values[2], name=f"{name}[2]"),
        )

    @classmethod
    def _coerce_duration(cls, duration) -> float:
        """Normalize camera animation duration with the same minimum as prior behavior."""
        return max(0.01, cls._coerce_finite_float(duration, name="duration"))

    @staticmethod
    def _zoom_from_z(z: float) -> float:
        """Map z-distance to visible world height with a floor for readability."""
        # In this 2D visualizer, z acts as camera distance, which maps to visible world height.
        return max(60.0, abs(Camera._coerce_finite_float(z, name="z")))

    def _sample(self, now: float | None = None):
        """Advance camera interpolation and return current `(x, y, z)` center."""
        if not self._moving:
            return self.center

        if now is None:
            now = time.perf_counter()

        progress = (now - self._start_time) / self._duration
        if progress >= 1.0:
            self.center = self._target_center
            self.height = self._target_height
            self._moving = False
            return self.center

        eased = _ease_progress(
            progress,
            windup_ratio=(self._windup / self._duration) if self._duration > 0.0 else 0.0,
            winddown_ratio=(self._winddown / self._duration) if self._duration > 0.0 else 0.0,
        )

        sx, sy, sz = self._start_center
        tx, ty, tz = self._target_center
        self.center = (
            sx + (tx - sx) * eased,
            sy + (ty - sy) * eased,
            sz + (tz - sz) * eased,
        )
        self.height = self._start_height + (self._target_height - self._start_height) * eased
        return self.center

    def move_to(
        self,
        x: float,
        y: float,
        z: float,
        duration: float = 1.2,
        *,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Start a camera move toward `(x, y, z)` over `duration` seconds and return `self`."""
        target_center = self._coerce_center((x, y, z), name="target center")
        current = self._sample()
        self.center = current
        self._start_center = current
        self._target_center = target_center
        self._start_height = float(self.height)
        self._target_height = self._zoom_from_z(z)
        self._start_time = time.perf_counter()
        self._duration = self._coerce_duration(duration)
        self._windup, self._winddown = _resolve_wind_timing(self._duration, windup=windup, winddown=winddown)
        self._moving = True
        return self

    def state(self):
        """Return serializable camera state as `{"center": [x, y, z], "height": h, "moving": bool}`."""
        cx, cy, cz = self._sample()
        return {
            "center": [cx, cy, cz],
            "height": float(self.height),
            "moving": bool(self._moving),
        }


canvas = Canvas()

camera = Camera()


_ANIMATION_COLOR_ALIASES = {
    "black": "#000000",
    "white": "#ffffff",
    "gray": "#808080",
    "grey": "#808080",
    "red": "#ff0000",
    "green": "#008000",
    "blue": "#0000ff",
    "yellow": "#ffff00",
    "orange": "#ffa500",
    "purple": "#800080",
    "pink": "#ffc0cb",
    "teal": "#008080",
    "cyan": "#00ffff",
    "amber": "#ffbf00",
}


def _finite_float(value: Any, *, name: str) -> float:
    """Coerce a value to a finite float or raise `ValueError` naming the offending field."""
    try:
        numeric = float(value)
    except Exception as exc:
        raise ValueError(f"{name} must be a finite number") from exc
    if not math.isfinite(numeric):
        raise ValueError(f"{name} must be a finite number")
    return numeric


def _normalize_duration(duration: Any) -> float:
    """Normalize animation duration to a finite float with a minimum of `0.01` seconds."""
    return max(0.01, _finite_float(duration, name="duration"))


def _smoothstep(progress: float) -> float:
    """Clamp progress to `[0, 1]` and apply smoothstep easing."""
    clamped = max(0.0, min(1.0, progress))
    return clamped * clamped * (3.0 - 2.0 * clamped)


def _normalize_wind_seconds(value: Any, *, name: str) -> float:
    """Normalize wind timing values to finite non-negative seconds."""
    seconds = _finite_float(value, name=name)
    if seconds < 0.0:
        raise ValueError(f"{name} must be >= 0")
    return seconds


def _resolve_wind_timing(duration: float, *, windup: Any, winddown: Any) -> tuple[float, float]:
    """Resolve windup/winddown durations in seconds and clamp them to total animation duration."""
    duration_value = _normalize_duration(duration)
    windup_value = _normalize_wind_seconds(windup, name="windup")
    winddown_value = _normalize_wind_seconds(winddown, name="winddown")
    total_wind = windup_value + winddown_value
    if total_wind <= duration_value:
        return windup_value, winddown_value
    if total_wind <= 0.0:
        return 0.0, 0.0
    scale = duration_value / total_wind
    return windup_value * scale, winddown_value * scale


def _normalize_wind_ratios(windup_ratio: float, winddown_ratio: float) -> tuple[float, float]:
    """Clamp wind ratios to `[0, 1]` and scale when combined range exceeds one full timeline."""
    up = max(0.0, min(1.0, float(windup_ratio)))
    down = max(0.0, min(1.0, float(winddown_ratio)))
    total = up + down
    if total <= 1.0:
        return up, down
    if total <= 0.0:
        return 0.0, 0.0
    scale = 1.0 / total
    return up * scale, down * scale


def _ease_progress(progress: float, *, windup_ratio: float, winddown_ratio: float) -> float:
    """Map linear progress to eased progress with configurable windup/winddown durations."""
    clamped = max(0.0, min(1.0, progress))
    up, down = _normalize_wind_ratios(windup_ratio, winddown_ratio)
    if up > 0.0 and clamped < up:
        x = clamped / up
        return up * (x * x * (2.0 - x))

    down_start = 1.0 - down
    if down > 0.0 and clamped > down_start:
        x = (clamped - down_start) / down
        return down_start + down * (x + (x * x) - (x * x * x))

    return clamped


def _wind_envelope(progress: float, *, windup_ratio: float, winddown_ratio: float) -> float:
    """Return amplitude envelope in `[0, 1]` with configurable windup/winddown boundary ramps."""
    clamped = max(0.0, min(1.0, progress))
    up, down = _normalize_wind_ratios(windup_ratio, winddown_ratio)
    if up > 0.0 and clamped < up:
        x = clamped / up
        return x * x * (2.0 - x)
    if down > 0.0 and clamped > (1.0 - down):
        x = (1.0 - clamped) / down
        return x * x * (2.0 - x)
    return 1.0


def _normalize_hex_color(raw_color: Any, *, name: str = "color") -> str:
    """Normalize color-like input to lowercase `#rrggbb`, validating aliases and hex format."""
    if raw_color is None:
        raise ValueError(f"{name} cannot be None")

    try:
        text = str(raw_color).strip().lower()
    except Exception as exc:
        raise ValueError(f"{name} must be a valid hex color string") from exc

    if not text:
        raise ValueError(f"{name} cannot be empty")

    text = _ANIMATION_COLOR_ALIASES.get(text, text)
    if not text.startswith("#"):
        raise ValueError(f"{name} must be a valid hex color string")

    hex_part = text[1:]
    if len(hex_part) == 3:
        if not all(ch in "0123456789abcdef" for ch in hex_part):
            raise ValueError(f"{name} must be a valid hex color string")
        hex_part = "".join(ch * 2 for ch in hex_part)
    elif len(hex_part) == 6:
        if not all(ch in "0123456789abcdef" for ch in hex_part):
            raise ValueError(f"{name} must be a valid hex color string")
    else:
        raise ValueError(f"{name} must be a valid hex color string")

    return f"#{hex_part}"


def _hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    """Convert a hex color string to an `(r, g, b)` tuple of integers."""
    normalized = _normalize_hex_color(hex_color)
    raw = normalized[1:]
    return (
        int(raw[0:2], 16),
        int(raw[2:4], 16),
        int(raw[4:6], 16),
    )


def _rgb_to_hex(rgb: tuple[float, float, float]) -> str:
    """Convert RGB channel values to a clamped lowercase `#rrggbb` string."""
    red, green, blue = rgb
    r = max(0, min(255, int(round(red))))
    g = max(0, min(255, int(round(green))))
    b = max(0, min(255, int(round(blue))))
    return f"#{r:02x}{g:02x}{b:02x}"


def _lerp_float(start: float, end: float, t: float) -> float:
    """Linearly interpolate between scalar values."""
    return start + (end - start) * t


def _lerp_color(start: str, end: str, t: float) -> str:
    """Linearly interpolate between two hex colors in RGB space."""
    start_rgb = _hex_to_rgb(start)
    end_rgb = _hex_to_rgb(end)
    return _rgb_to_hex(
        (
            _lerp_float(start_rgb[0], end_rgb[0], t),
            _lerp_float(start_rgb[1], end_rgb[1], t),
            _lerp_float(start_rgb[2], end_rgb[2], t),
        )
    )


@dataclass
class _PropertyAnimation:
    """Generic interpolation record for a single animated metadata property."""

    start: Any
    end: Any
    start_time: float
    duration: float
    windup: float
    winddown: float
    interpolate: Callable[[Any, Any, float], Any]
    apply: Callable[[Any], None]

    def sample(self, now: float) -> bool:
        """Apply interpolated value at `now`; return `True` once animation reaches its target."""
        progress = (now - self.start_time) / self.duration
        if progress >= 1.0:
            self.apply(self.end)
            return True
        if progress <= 0.0:
            self.apply(self.start)
            return False

        eased = _ease_progress(
            progress,
            windup_ratio=(self.windup / self.duration) if self.duration > 0.0 else 0.0,
            winddown_ratio=(self.winddown / self.duration) if self.duration > 0.0 else 0.0,
        )
        self.apply(self.interpolate(self.start, self.end, eased))
        return False


@dataclass
class _HighlightAnimation:
    """Transient style overlay that eases into and out of emphasized values."""

    scalar_channels: dict[str, tuple[float, float]]
    color_channels: dict[str, tuple[str, str]]
    start_time: float
    duration: float
    cycles: int
    windup: float
    winddown: float


class _StyleAnimationMixin:
    """Shared helpers for scalar/color metadata animations on nodes and edges."""

    _animations: dict[str, _PropertyAnimation]
    _highlight: _HighlightAnimation | None

    def _animation_metadata(self):
        """Return the metadata object that owns animated style fields."""
        return self.metadata

    def _read_scalar_metadata(self, attr_name: str, *, min_value: float = 0.0) -> float:
        """Read and sanitize a numeric metadata field, persisting the normalized value."""
        metadata = self._animation_metadata()
        raw = getattr(metadata, attr_name, min_value)
        try:
            numeric = float(raw)
        except Exception:
            numeric = min_value
        if not math.isfinite(numeric):
            numeric = min_value
        numeric = max(min_value, numeric)
        setattr(metadata, attr_name, numeric)
        return numeric

    def _read_color_metadata(self, attr_name: str, *, fallback: str) -> str:
        """Read and normalize a color metadata field, falling back to a valid default."""
        metadata = self._animation_metadata()
        raw = getattr(metadata, attr_name, fallback)
        try:
            normalized = _normalize_hex_color(raw, name=attr_name)
        except ValueError:
            normalized = _normalize_hex_color(fallback, name=attr_name)
        setattr(metadata, attr_name, normalized)
        return normalized

    def _set_scalar_metadata(self, attr_name: str, value: float, *, min_value: float = 0.0) -> float:
        """Validate and assign a numeric metadata field; return stored value."""
        numeric = _finite_float(value, name=attr_name)
        if numeric < min_value:
            raise ValueError(f"{attr_name} must be >= {min_value}")
        metadata = self._animation_metadata()
        setattr(metadata, attr_name, numeric)
        return numeric

    def _set_color_metadata(self, attr_name: str, value) -> str:
        """Validate and assign a color metadata field; return normalized hex color."""
        normalized = _normalize_hex_color(value, name=attr_name)
        metadata = self._animation_metadata()
        setattr(metadata, attr_name, normalized)
        return normalized

    def _start_scalar_animation(
        self,
        attr_name: str,
        target,
        duration: float,
        *,
        now: float | None = None,
        min_value: float = 0.0,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Start/restart scalar animation for `attr_name`; stores a `_PropertyAnimation` in `_animations`."""
        if now is None:
            now = time.perf_counter()

        duration_value = _normalize_duration(duration)
        resolved_windup, resolved_winddown = _resolve_wind_timing(
            duration_value,
            windup=windup,
            winddown=winddown,
        )

        self._cancel_highlight_attr(attr_name)
        current = self._read_scalar_metadata(attr_name, min_value=min_value)
        target_value = _finite_float(target, name=attr_name)
        if target_value < min_value:
            raise ValueError(f"{attr_name} must be >= {min_value}")

        self._animations.pop(attr_name, None)
        if math.isclose(current, target_value, rel_tol=0.0, abs_tol=1e-9):
            self._set_scalar_metadata(attr_name, target_value, min_value=min_value)
            return self

        self._animations[attr_name] = _PropertyAnimation(
            start=current,
            end=target_value,
            start_time=now,
            duration=duration_value,
            windup=resolved_windup,
            winddown=resolved_winddown,
            interpolate=_lerp_float,
            apply=lambda value, key=attr_name, lower=min_value: self._set_scalar_metadata(key, value, min_value=lower),
        )
        return self

    def _start_color_animation(
        self,
        attr_name: str,
        target,
        duration: float,
        *,
        now: float | None = None,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Start/restart color animation for `attr_name` toward normalized `target`."""
        if now is None:
            now = time.perf_counter()

        duration_value = _normalize_duration(duration)
        resolved_windup, resolved_winddown = _resolve_wind_timing(
            duration_value,
            windup=windup,
            winddown=winddown,
        )

        self._cancel_highlight_attr(attr_name)
        target_value = _normalize_hex_color(target, name=attr_name)
        current = self._read_color_metadata(attr_name, fallback=target_value)

        self._animations.pop(attr_name, None)
        if current == target_value:
            self._set_color_metadata(attr_name, target_value)
            return self

        self._animations[attr_name] = _PropertyAnimation(
            start=current,
            end=target_value,
            start_time=now,
            duration=duration_value,
            windup=resolved_windup,
            winddown=resolved_winddown,
            interpolate=_lerp_color,
            apply=lambda value, key=attr_name: self._set_color_metadata(key, value),
        )
        return self

    def _cancel_highlight_attr(self, attr_name: str):
        """Cancel in-flight highlight when a direct animation targets the same style field."""
        highlight = self._highlight
        if highlight is None:
            return
        if attr_name in highlight.scalar_channels or attr_name in highlight.color_channels:
            self._highlight = None

    @staticmethod
    def _highlight_profile(progress: float, *, cycles: int) -> float:
        """Return smooth oscillation intensity in `[0, 1]` with zero slope at boundaries."""
        clamped = max(0.0, min(1.0, progress))
        return 0.5 - 0.5 * math.cos((2.0 * math.pi * max(1, cycles)) * clamped)

    def _start_highlight_animation(
        self,
        *,
        scalar_attrs: Iterable[str],
        color_attrs: Iterable[str],
        duration: float,
        scale: float = 1.2,
        color: str | None = None,
        pulse: bool = False,
        windup: float = 0.3,
        winddown: float = 0.3,
        now: float | None = None,
        min_value: float = 0.0,
    ):
        """Start/restart a temporary highlight overlay for selected scalar/color metadata fields."""
        if now is None:
            now = time.perf_counter()

        duration_value = _normalize_duration(duration)
        resolved_windup, resolved_winddown = _resolve_wind_timing(
            duration_value,
            windup=windup,
            winddown=winddown,
        )
        scale_value = _finite_float(scale, name="scale")
        if scale_value <= 0.0:
            raise ValueError("scale must be > 0")
        highlight_color = None if color is None else _normalize_hex_color(color, name="color")
        pulse_enabled = bool(pulse)

        scalar_keys = tuple(dict.fromkeys(str(attr) for attr in scalar_attrs if attr is not None))
        color_keys = tuple(dict.fromkeys(str(attr) for attr in color_attrs if attr is not None))
        if not scalar_keys and not color_keys:
            raise ValueError("highlight requires at least one scalar or color attribute")

        for attr_name in (*scalar_keys, *color_keys):
            self._animations.pop(attr_name, None)

        scalar_channels: dict[str, tuple[float, float]] = {}
        for attr_name in scalar_keys:
            current_value = self._read_scalar_metadata(attr_name, min_value=min_value)
            scalar_channels[attr_name] = (current_value, current_value * scale_value)

        color_channels: dict[str, tuple[str, str]] = {}
        for attr_name in color_keys:
            current_color = self._read_color_metadata(attr_name, fallback="#000000")
            target_color = current_color if highlight_color is None else highlight_color
            color_channels[attr_name] = (current_color, target_color)

        unchanged_scalars = all(math.isclose(start, end, rel_tol=0.0, abs_tol=1e-9) for start, end in scalar_channels.values())
        unchanged_colors = all(start == end for start, end in color_channels.values())
        if unchanged_scalars and unchanged_colors:
            self._highlight = None
            return self

        cycles = 1
        if pulse_enabled:
            cycles = max(2, int(round(duration_value / 1.8)))

        self._highlight = _HighlightAnimation(
            scalar_channels=scalar_channels,
            color_channels=color_channels,
            start_time=now,
            duration=duration_value,
            cycles=cycles,
            windup=resolved_windup,
            winddown=resolved_winddown,
        )
        return self

    def _sample_highlight(self, now: float) -> bool:
        """Advance highlight overlay and restore base values when finished."""
        highlight = self._highlight
        if highlight is None:
            return False

        progress = (now - highlight.start_time) / highlight.duration
        if progress >= 1.0:
            for attr_name, (base_value, _target_value) in highlight.scalar_channels.items():
                self._set_scalar_metadata(attr_name, base_value, min_value=0.0)
            for attr_name, (base_color, _target_color) in highlight.color_channels.items():
                self._set_color_metadata(attr_name, base_color)
            self._highlight = None
            return False

        phase = _ease_progress(
            progress,
            windup_ratio=(highlight.windup / highlight.duration) if highlight.duration > 0.0 else 0.0,
            winddown_ratio=(highlight.winddown / highlight.duration) if highlight.duration > 0.0 else 0.0,
        )
        intensity = self._highlight_profile(phase, cycles=highlight.cycles)
        for attr_name, (base_value, target_value) in highlight.scalar_channels.items():
            self._set_scalar_metadata(attr_name, _lerp_float(base_value, target_value, intensity), min_value=0.0)
        for attr_name, (base_color, target_color) in highlight.color_channels.items():
            self._set_color_metadata(attr_name, _lerp_color(base_color, target_color, intensity))
        return True

    def _sample_style_animations(self, now: float) -> bool:
        """Advance scalar/color animations and transient highlights."""
        if self._animations:
            # Iterate over a snapshot so completed animations can be removed mid-loop.
            for attr_name, animation in list(self._animations.items()):
                completed = animation.sample(now)
                if completed:
                    self._animations.pop(attr_name, None)

        highlight_active = self._sample_highlight(now)
        return bool(self._animations or highlight_active)

    def _style_moving(self) -> bool:
        """Return whether any style animations are still in-flight."""
        return bool(self._animations or self._highlight is not None)


@dataclass
class NodeMeta:
    """Per-node style settings consumed by renderers and style animations."""

    fill_color: str = "#e8c547"
    stroke_color: str = "#1f2937"
    stroke_width: float = 3.0
    diameter: float = 50.0
    label_size: float = 30.0
    label_color: str = "#111827"
    display_label: bool = True


@dataclass
class Node(_StyleAnimationMixin):
    """Graph node with position, style metadata, and optional motion animations."""

    id: str
    label: str | None = None
    metadata: NodeMeta = field(default_factory=NodeMeta)
    pos: tuple[float, float] | None = None
    _start_pos: tuple[float, float] = field(default=(0.0, 0.0), init=False, repr=False)
    _target_pos: tuple[float, float] = field(default=(0.0, 0.0), init=False, repr=False)
    _start_time: float = field(default=0.0, init=False, repr=False)
    _duration: float = field(default=0.0, init=False, repr=False)
    _pos_windup: float = field(default=0.0, init=False, repr=False)
    _pos_winddown: float = field(default=0.0, init=False, repr=False)
    _pos_moving: bool = field(default=False, init=False, repr=False)
    _moving: bool = field(default=False, init=False, repr=False)
    _wiggle_start_pos: tuple[float, float] = field(default=(0.0, 0.0), init=False, repr=False)
    _wiggle_start_time: float = field(default=0.0, init=False, repr=False)
    _wiggle_duration: float = field(default=0.0, init=False, repr=False)
    _wiggle_speed: float = field(default=0.0, init=False, repr=False)
    _wiggle_temperature: float = field(default=0.0, init=False, repr=False)
    _wiggle_phase_x: float = field(default=0.0, init=False, repr=False)
    _wiggle_phase_y: float = field(default=0.0, init=False, repr=False)
    _wiggle_freq_x: float = field(default=1.0, init=False, repr=False)
    _wiggle_freq_y: float = field(default=1.0, init=False, repr=False)
    _wiggle_windup: float = field(default=0.0, init=False, repr=False)
    _wiggle_winddown: float = field(default=0.0, init=False, repr=False)
    _wiggle_active: bool = field(default=False, init=False, repr=False)
    _animations: dict[str, _PropertyAnimation] = field(default_factory=dict, init=False, repr=False)
    _highlight: _HighlightAnimation | None = field(default=None, init=False, repr=False)

    def __post_init__(self):
        """Normalize identifiers/labels and validate constructor-provided metadata/position."""
        if not isinstance(self.metadata, NodeMeta):
            raise TypeError("metadata must be a NodeMeta instance")
        if self.pos is not None and self._parse_pos(self.pos) is None:
            raise ValueError("pos must be None or contain finite x/y coordinates")

        node_id = str(self.id)
        if not node_id.strip():
            raise ValueError("node id cannot be empty")
        self.id = node_id
        if self.label is None:
            self.label = self.id
        else:
            self.label = str(self.label)

    @staticmethod
    def _parse_pos(raw_pos):
        """Parse `(x, y)` from tuple/list/dict inputs; return `None` when invalid."""
        if isinstance(raw_pos, (list, tuple)) and len(raw_pos) >= 2:
            try:
                x = float(raw_pos[0])
                y = float(raw_pos[1])
            except Exception:
                return None
            if math.isfinite(x) and math.isfinite(y):
                return (x, y)
            return None

        if isinstance(raw_pos, dict):
            try:
                x = float(raw_pos.get("x"))
                y = float(raw_pos.get("y"))
            except Exception:
                return None
            if math.isfinite(x) and math.isfinite(y):
                return (x, y)
            return None

        return None

    @staticmethod
    def _seed_unit(text: str, salt: int) -> float:
        """Generate deterministic pseudo-random unit value from node identity and salt."""
        value = 0
        for index, char in enumerate(f"{text}:{salt}"):
            value = (value * 131 + (index + salt + 1) * ord(char)) % 104729
        return value / 104729.0

    def _start_wiggle(
        self,
        speed: float,
        duration: float,
        temperature: float,
        *,
        windup: float = 0.7,
        winddown: float = 0.7,
        now: float | None = None,
    ):
        """Configure deterministic wiggle animation; returns `self` for chaining."""
        speed_value = _finite_float(speed, name="speed")
        if speed_value <= 0.0:
            raise ValueError("speed must be > 0")
        duration_value = _normalize_duration(duration)
        resolved_windup, resolved_winddown = _resolve_wind_timing(
            duration_value,
            windup=windup,
            winddown=winddown,
        )
        temperature_value = _finite_float(temperature, name="temperature")
        if temperature_value < 0.0 or temperature_value > 1.0:
            raise ValueError("temperature must be between 0 and 1 inclusive")

        if now is None:
            now = time.perf_counter()

        current = self._sample(now=now)
        if current is None:
            current = self._parse_pos(self.pos)
        if current is None:
            current = self._target_pos
            self.pos = current

        node_identity = str(self.id)
        # Tie phase/frequency to node id so each node has stable motion characteristics.
        self._wiggle_phase_x = 2.0 * math.pi * self._seed_unit(node_identity, 11)
        self._wiggle_phase_y = 2.0 * math.pi * self._seed_unit(node_identity, 29)
        self._wiggle_freq_x = 0.8 + (1.6 * self._seed_unit(node_identity, 47))
        self._wiggle_freq_y = 0.8 + (1.6 * self._seed_unit(node_identity, 83))
        self._wiggle_start_pos = current
        self._wiggle_start_time = now
        self._wiggle_duration = duration_value
        self._wiggle_speed = speed_value
        self._wiggle_temperature = temperature_value
        self._wiggle_windup = resolved_windup
        self._wiggle_winddown = resolved_winddown
        self._wiggle_active = temperature_value > 0.0
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def _sample_wiggle(self, now: float, current_pos: tuple[float, float] | None):
        """Sample wiggle offset for `current_pos`; returns `((x, y) | None, is_active)`."""
        if not self._wiggle_active:
            return current_pos, False
        if current_pos is None:
            return current_pos, False

        progress = (now - self._wiggle_start_time) / self._wiggle_duration
        if progress >= 1.0:
            self._wiggle_active = False
            self.pos = self._wiggle_start_pos
            return self._wiggle_start_pos, False

        clamped_progress = max(0.0, min(1.0, progress))
        envelope = _wind_envelope(
            clamped_progress,
            windup_ratio=(self._wiggle_windup / self._wiggle_duration) if self._wiggle_duration > 0.0 else 0.0,
            winddown_ratio=(self._wiggle_winddown / self._wiggle_duration) if self._wiggle_duration > 0.0 else 0.0,
        )
        phase_time = (now - self._wiggle_start_time) * self._wiggle_speed

        chaos = self._wiggle_temperature
        detail_mix = 0.25 + (0.55 * chaos)
        amplitude = (5.0 + (15.0 * chaos)) * chaos

        wave_x = math.sin((phase_time * self._wiggle_freq_x) + self._wiggle_phase_x)
        wave_x += detail_mix * math.sin((phase_time * (self._wiggle_freq_x * (1.7 + chaos))) + (self._wiggle_phase_y * 0.7))
        wave_y = math.cos((phase_time * self._wiggle_freq_y) + self._wiggle_phase_y)
        wave_y += detail_mix * math.sin((phase_time * (self._wiggle_freq_y * (2.1 + chaos))) + (self._wiggle_phase_x * 1.3))

        offset_scale = amplitude * envelope
        return (
            self._wiggle_start_pos[0] + (wave_x * offset_scale),
            self._wiggle_start_pos[1] + (wave_y * offset_scale),
        ), True

    def _sample(self, now: float | None = None):
        """Advance position/style/wiggle animations and return current `(x, y)` or `None`."""
        if now is None:
            now = time.perf_counter()

        current_pos = self._parse_pos(self.pos)
        if self._pos_moving:
            progress = (now - self._start_time) / self._duration
            if progress >= 1.0:
                current_pos = self._target_pos
                self.pos = current_pos
                self._pos_moving = False
            elif progress <= 0.0:
                current_pos = self._start_pos
                self.pos = current_pos
            else:
                eased = _ease_progress(
                    progress,
                    windup_ratio=(self._pos_windup / self._duration) if self._duration > 0.0 else 0.0,
                    winddown_ratio=(self._pos_winddown / self._duration) if self._duration > 0.0 else 0.0,
                )
                sx, sy = self._start_pos
                tx, ty = self._target_pos
                current_pos = (
                    sx + (tx - sx) * eased,
                    sy + (ty - sy) * eased,
                )
                self.pos = current_pos

        current_pos, wiggle_moving = self._sample_wiggle(now, current_pos)
        style_moving = self._sample_style_animations(now)
        self._moving = bool(self._pos_moving or style_moving or wiggle_moving)
        return current_pos

    def move_to(
        self,
        x: float,
        y: float,
        duration: float = 1.2,
        *,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Animate node position toward `(x, y)` over `duration` seconds; returns `self`."""
        target = (
            _finite_float(x, name="x"),
            _finite_float(y, name="y"),
        )

        now = time.perf_counter()
        current = self._sample(now=now)
        self._wiggle_active = False
        if current is None:
            current = self._parse_pos(self.pos)
        if current is None:
            current = self._target_pos
            self.pos = current

        self._start_pos = current
        self._target_pos = target
        self._start_time = now
        self._duration = _normalize_duration(duration)
        self._pos_windup, self._pos_winddown = _resolve_wind_timing(
            self._duration,
            windup=windup,
            winddown=winddown,
        )
        self._pos_moving = self._start_pos != self._target_pos
        if not self._pos_moving:
            self.pos = self._target_pos

        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def change_diameter(
        self,
        to,
        duration: float = 1.2,
        *,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Animate `metadata.diameter` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_scalar_animation(
            "diameter",
            to,
            duration,
            now=now,
            min_value=0.0,
            windup=windup,
            winddown=winddown,
        )
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def change_fill_color(
        self,
        to,
        duration: float = 1.2,
        *,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Animate `metadata.fill_color` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_color_animation(
            "fill_color",
            to,
            duration,
            now=now,
            windup=windup,
            winddown=winddown,
        )
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def change_stroke_color(
        self,
        to,
        duration: float = 1.2,
        *,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Animate `metadata.stroke_color` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_color_animation(
            "stroke_color",
            to,
            duration,
            now=now,
            windup=windup,
            winddown=winddown,
        )
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def change_stroke_width(
        self,
        to,
        duration: float = 1.2,
        *,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Animate `metadata.stroke_width` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_scalar_animation(
            "stroke_width",
            to,
            duration,
            now=now,
            min_value=0.0,
            windup=windup,
            winddown=winddown,
        )
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def change_label_size(
        self,
        to,
        duration: float = 1.2,
        *,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Animate `metadata.label_size` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_scalar_animation(
            "label_size",
            to,
            duration,
            now=now,
            min_value=0.0,
            windup=windup,
            winddown=winddown,
        )
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def scale(
        self,
        factor,
        duration: float = 1.2,
        *,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Scale diameter/stroke/label sizes together by `factor` over `duration`; returns `self`."""
        factor_value = _finite_float(factor, name="factor")
        if factor_value <= 0.0:
            raise ValueError("factor must be > 0")

        now = time.perf_counter()
        self._sample(now=now)
        duration_value = _normalize_duration(duration)

        current_diameter = self._read_scalar_metadata("diameter", min_value=0.0)
        current_stroke_width = self._read_scalar_metadata("stroke_width", min_value=0.0)
        current_label_size = self._read_scalar_metadata("label_size", min_value=0.0)

        self._start_scalar_animation(
            "diameter",
            current_diameter * factor_value,
            duration_value,
            now=now,
            min_value=0.0,
            windup=windup,
            winddown=winddown,
        )
        self._start_scalar_animation(
            "stroke_width",
            current_stroke_width * factor_value,
            duration_value,
            now=now,
            min_value=0.0,
            windup=windup,
            winddown=winddown,
        )
        self._start_scalar_animation(
            "label_size",
            current_label_size * factor_value,
            duration_value,
            now=now,
            min_value=0.0,
            windup=windup,
            winddown=winddown,
        )
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def highlight(
        self,
        duration: float = 1.8,
        scale: float = 1.2,
        color: str | None = None,
        pulse: bool = False,
        *,
        windup: float = 0.3,
        winddown: float = 0.3,
    ):
        """Temporarily emphasize node size (and optional color) then ease back to baseline."""
        now = time.perf_counter()
        self._sample(now=now)
        color_attrs = ("fill_color",) if color is not None else ()
        self._start_highlight_animation(
            scalar_attrs=("diameter", "stroke_width", "label_size"),
            color_attrs=color_attrs,
            duration=duration,
            scale=scale,
            color=color,
            pulse=pulse,
            windup=windup,
            winddown=winddown,
            now=now,
            min_value=0.0,
        )
        self._moving = bool(self._pos_moving or self._style_moving() or self._wiggle_active)
        return self

    def state(self):
        """Return serializable node state dict with position, style fields, and `moving` flag."""
        now = time.perf_counter()
        pos = self._sample(now=now)
        return {
            "pos": [pos[0], pos[1]] if pos is not None else None,
            "fill_color": self._read_color_metadata("fill_color", fallback="#e8c547"),
            "stroke_color": self._read_color_metadata("stroke_color", fallback="#1f2937"),
            "stroke_width": self._read_scalar_metadata("stroke_width", min_value=0.0),
            "diameter": self._read_scalar_metadata("diameter", min_value=0.0),
            "label_size": self._read_scalar_metadata("label_size", min_value=0.0),
            "label_color": self._read_color_metadata("label_color", fallback="#111827"),
            "moving": bool(self._moving),
        }


@dataclass
class EdgeMeta:
    """Per-edge style settings including path hints consumed by renderers and style animations."""

    color: str = "#8a8a8a"
    width: float = 4.0
    display_label: bool = False
    path_style: str = "straight"
    path_params: dict[str, Any] = field(default_factory=dict)


@dataclass
class Edge(_StyleAnimationMixin):
    """Directed connection between two nodes with animated style metadata and path configuration."""

    frm: Node
    to: Node
    label: str = ""
    metadata: EdgeMeta = field(default_factory=EdgeMeta)
    path_fn: Callable[["Edge"], Any] | None = None
    _moving: bool = field(default=False, init=False, repr=False)
    _animations: dict[str, _PropertyAnimation] = field(default_factory=dict, init=False, repr=False)
    _highlight: _HighlightAnimation | None = field(default=None, init=False, repr=False)

    _DEFAULT_PATH_STYLE = "straight"
    _BUILTIN_PATH_STYLES = frozenset({"straight", "bezier", "arc", "orthogonal", "polyline"})

    def __post_init__(self):
        """Validate endpoints/metadata and normalize optional label/path configuration."""
        if not isinstance(self.frm, Node):
            raise TypeError("frm must be a Node instance")
        if not isinstance(self.to, Node):
            raise TypeError("to must be a Node instance")
        if not isinstance(self.metadata, EdgeMeta):
            raise TypeError("metadata must be an EdgeMeta instance")
        if self.label is None:
            self.label = ""
        elif not isinstance(self.label, str):
            self.label = str(self.label)
        if self.path_fn is not None and not callable(self.path_fn):
            raise TypeError("path_fn must be callable or None")

        base_path = self._normalize_path_spec(
            {
                "style": self.metadata.path_style,
                "params": self.metadata.path_params,
            },
            name="metadata path spec",
            allow_custom_style=False,
        )
        self.metadata.path_style = base_path["style"]
        self.metadata.path_params = base_path["params"]

    @classmethod
    def _normalize_path_style(cls, raw_style: Any, *, name: str, allow_custom: bool) -> str:
        """Normalize path style names; supports built-ins and optional custom extension styles."""
        if raw_style is None:
            return cls._DEFAULT_PATH_STYLE
        style = str(raw_style).strip().lower()
        if not style:
            raise ValueError(f"{name} must be a non-empty string")
        if not allow_custom and style not in cls._BUILTIN_PATH_STYLES:
            supported = ", ".join(sorted(cls._BUILTIN_PATH_STYLES))
            raise ValueError(f"{name} must be one of: {supported}")
        return style

    @classmethod
    def _normalize_json_like(cls, value: Any, *, name: str):
        """Recursively coerce path payload values to JSON-like structures with finite numbers."""
        if value is None or isinstance(value, (str, bool)):
            return value
        if isinstance(value, int):
            return int(value)
        if isinstance(value, float):
            if not math.isfinite(value):
                raise ValueError(f"{name} must contain only finite numbers")
            return float(value)
        if isinstance(value, (list, tuple)):
            return [cls._normalize_json_like(item, name=f"{name}[{index}]") for index, item in enumerate(value)]
        if isinstance(value, Mapping):
            normalized = {}
            for key, item in value.items():
                if not isinstance(key, str):
                    raise TypeError(f"{name} keys must be strings")
                normalized[key] = cls._normalize_json_like(item, name=f"{name}.{key}")
            return normalized
        raise TypeError(
            f"{name} contains unsupported value type {type(value).__name__}; "
            "expected JSON-like scalars, lists, or dicts"
        )

    @classmethod
    def _normalize_path_params(cls, raw_params: Any, *, name: str = "path params") -> dict[str, Any]:
        """Validate path parameter payload and return a JSON-serializable dictionary."""
        if raw_params is None:
            return {}
        if not isinstance(raw_params, Mapping):
            raise TypeError(f"{name} must be a mapping/dict")
        normalized: dict[str, Any] = {}
        for key, value in raw_params.items():
            if not isinstance(key, str):
                raise TypeError(f"{name} keys must be strings")
            normalized[key] = cls._normalize_json_like(value, name=f"{name}.{key}")
        return normalized

    @classmethod
    def _normalize_path_spec(
        cls,
        raw_spec: Any,
        *,
        name: str,
        allow_custom_style: bool,
    ) -> dict[str, Any]:
        """Normalize a path spec to `{"style": str, "params": dict}` with alias support."""
        if isinstance(raw_spec, str):
            spec: dict[str, Any] = {"style": raw_spec, "params": {}}
        elif isinstance(raw_spec, Mapping):
            spec = dict(raw_spec)
        else:
            raise TypeError(f"{name} must be a style string or mapping/dict")

        style_raw = spec.get("style")
        if style_raw is None:
            style_raw = spec.get("path_style")
        if style_raw is None:
            style_raw = spec.get("pathStyle")

        style = cls._normalize_path_style(
            style_raw if style_raw is not None else cls._DEFAULT_PATH_STYLE,
            name=f"{name}.style",
            allow_custom=allow_custom_style,
        )

        params_raw = None
        params_name = "params"
        for params_key in ("params", "path_params", "pathParams", "spec", "path_spec", "pathSpec"):
            if params_key in spec:
                params_raw = spec[params_key]
                params_name = params_key
                break
        params = cls._normalize_path_params(params_raw, name=f"{name}.{params_name}")

        normalized_spec: dict[str, Any] = {"style": style, "params": params}
        ignored_keys = {
            "style",
            "path_style",
            "pathStyle",
            "params",
            "path_params",
            "pathParams",
            "spec",
            "path_spec",
            "pathSpec",
        }
        for key, value in spec.items():
            if key in ignored_keys:
                continue
            if not isinstance(key, str):
                raise TypeError(f"{name} keys must be strings")
            normalized_spec[key] = cls._normalize_json_like(value, name=f"{name}.{key}")
        return normalized_spec

    @classmethod
    def _merge_path_specs(cls, base_spec: dict[str, Any], override_spec: Any) -> Any:
        """Merge mapping overrides onto a base path spec while preserving defaults."""
        if not isinstance(override_spec, Mapping):
            return override_spec

        override = dict(override_spec)
        merged: dict[str, Any] = dict(base_spec)
        merged.update(override)

        if "style" not in override:
            if "path_style" in override:
                merged["style"] = override["path_style"]
            elif "pathStyle" in override:
                merged["style"] = override["pathStyle"]

        params_key = None
        for candidate in ("params", "path_params", "pathParams", "spec", "path_spec", "pathSpec"):
            if candidate in override:
                params_key = candidate
                break

        base_params = base_spec.get("params", {})
        if params_key is None:
            merged["params"] = dict(base_params) if isinstance(base_params, Mapping) else base_params
            return merged

        override_params = override[params_key]
        if isinstance(base_params, Mapping) and isinstance(override_params, Mapping):
            merged_params = dict(base_params)
            merged_params.update(dict(override_params))
            merged["params"] = merged_params
            return merged

        merged["params"] = override_params
        return merged

    def resolve_path_spec(self) -> dict[str, Any]:
        """Return the normalized serializable path spec used by the visualizer for this edge."""
        base_spec = self._normalize_path_spec(
            {
                "style": self.metadata.path_style,
                "params": self.metadata.path_params,
            },
            name="metadata path spec",
            allow_custom_style=False,
        )

        if self.path_fn is None:
            return base_spec

        try:
            custom_spec = self.path_fn(self)
        except Exception as exc:
            raise ValueError(f"path_fn failed for edge '{self.frm.id}->{self.to.id}'") from exc

        if custom_spec is None:
            return base_spec

        merged_spec = self._merge_path_specs(base_spec, custom_spec)
        return self._normalize_path_spec(
            merged_spec,
            name="path_fn result",
            allow_custom_style=True,
        )

    def _sample(self, now: float | None = None):
        """Advance style animation state and return `self`."""
        if now is None:
            now = time.perf_counter()
        self._moving = bool(self._sample_style_animations(now))
        return self

    def change_width(
        self,
        to,
        duration: float = 1.2,
        *,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Animate `metadata.width` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_scalar_animation(
            "width",
            to,
            duration,
            now=now,
            min_value=0.0,
            windup=windup,
            winddown=winddown,
        )
        self._moving = bool(self._style_moving())
        return self

    def change_color(
        self,
        to,
        duration: float = 1.2,
        *,
        windup: float = 0.2,
        winddown: float = 0.2,
    ):
        """Animate `metadata.color` to `to` over `duration` seconds; returns `self`."""
        now = time.perf_counter()
        self._sample(now=now)
        self._start_color_animation(
            "color",
            to,
            duration,
            now=now,
            windup=windup,
            winddown=winddown,
        )
        self._moving = bool(self._style_moving())
        return self

    def highlight(
        self,
        duration: float = 1.8,
        scale: float = 1.2,
        color: str | None = None,
        pulse: bool = False,
        *,
        windup: float = 0.3,
        winddown: float = 0.3,
    ):
        """Temporarily emphasize edge width (and optional color) then ease back to baseline."""
        now = time.perf_counter()
        self._sample(now=now)
        color_attrs = ("color",) if color is not None else ()
        self._start_highlight_animation(
            scalar_attrs=("width",),
            color_attrs=color_attrs,
            duration=duration,
            scale=scale,
            color=color,
            pulse=pulse,
            windup=windup,
            winddown=winddown,
            now=now,
            min_value=0.0,
        )
        self._moving = bool(self._style_moving())
        return self

    def state(self):
        """Return serializable edge style/path state with color, width, path spec, and moving flag."""
        now = time.perf_counter()
        self._sample(now=now)
        return {
            "color": self._read_color_metadata("color", fallback="#8a8a8a"),
            "width": self._read_scalar_metadata("width", min_value=0.0),
            "path": self.resolve_path_spec(),
            "moving": bool(self._moving),
        }


@dataclass
class GraphMeta:
    """Top-level graph metadata for labels/layout hints in renderers."""

    label: str = ""
    layout: str = "force"
    display_label: bool = False


class Graph:
    """Container for nodes/edges with coercion helpers and graph-wide animation controls."""

    def __init__(self, nodes, edges, metadata=None):
        """Build a graph from node/edge iterables, coercing inputs through add helpers."""
        self.metadata = metadata if isinstance(metadata, GraphMeta) else GraphMeta()
        self.nodes = []
        self.edges = []
        self._nodes_by_id = {}
        self._nodes_by_identity = {}

        for node in self._iter_or_raise(nodes, name="nodes"):
            self.add_node(node)

        for edge in self._iter_or_raise(edges, name="edges"):
            self.add_edge(edge)

    @staticmethod
    def _iter_or_raise(values, *, name: str):
        """Return an iterator for constructor inputs with explicit `None`/iterability checks."""
        if values is None:
            raise ValueError(f"{name} cannot be None")
        try:
            return iter(values)
        except TypeError as exc:
            raise TypeError(f"{name} must be an iterable") from exc

    @staticmethod
    def _node_id_key(node_id) -> str:
        """Canonicalize node identifiers for dictionary-based lookup."""
        return str(node_id)

    def _register_node(self, node: Node):
        """Insert a canonical node once and synchronize identity/id indices."""
        self.nodes.append(node)
        self._nodes_by_identity[id(node)] = node
        self._nodes_by_id.setdefault(self._node_id_key(node.id), node)
        return node

    def _find_node_by_identity(self, candidate):
        """Return canonical node by object identity when available, else `None`."""
        indexed = self._nodes_by_identity.get(id(candidate))
        if indexed is candidate:
            return indexed

        # Fallback scan keeps indices resilient if callers mutate `self.nodes` directly.
        for existing in self.nodes:
            if existing is candidate:
                self._nodes_by_identity[id(existing)] = existing
                self._nodes_by_id.setdefault(self._node_id_key(existing.id), existing)
                return existing
        return None

    def _find_node_by_id(self, node_id):
        """Return first node whose stringified id matches `node_id`, else `None`."""
        key = self._node_id_key(node_id)
        indexed = self._nodes_by_id.get(key)
        if indexed is not None:
            return indexed

        # Preserve historical behavior if external code mutates `self.nodes`.
        for candidate in self.nodes:
            if self._node_id_key(candidate.id) == key:
                self._nodes_by_id.setdefault(key, candidate)
                self._nodes_by_identity[id(candidate)] = candidate
                return candidate
        return None

    def _coerce_node(self, node):
        """Coerce shorthand node inputs into a `Node` instance without inserting it."""
        if isinstance(node, Node):
            return node
        if node is None:
            raise ValueError("node cannot be None")
        if isinstance(node, tuple):
            if len(node) != 2:
                raise ValueError("node tuple shorthand must be (id, label)")
            node_id, node_label = node
            return Node(id=self._node_id_key(node_id), label=None if node_label is None else str(node_label))
        return Node(id=self._node_id_key(node))

    def _ensure_node(self, node):
        """Return canonical node object in the graph, matching by identity first then id."""
        candidate = self._coerce_node(node)

        by_identity = self._find_node_by_identity(candidate)
        if by_identity is not None:
            return by_identity

        # Fall back to id-based matching so shorthand inputs reuse existing logical nodes.
        by_id = self._find_node_by_id(candidate.id)
        if by_id is not None:
            return by_id

        return self._register_node(candidate)

    def _coerce_edge(self, edge, anchor_node=None):
        """Coerce edge shorthands into an `Edge`, reusing canonical endpoint node objects."""
        if edge is None:
            raise ValueError("edge cannot be None")

        if anchor_node is not None:
            anchor_node = self._ensure_node(anchor_node)

        if isinstance(edge, Edge):
            frm = self._ensure_node(edge.frm)
            to = self._ensure_node(edge.to)
            if frm is edge.frm and to is edge.to:
                return edge
            return Edge(frm=frm, to=to, label=edge.label, metadata=edge.metadata, path_fn=edge.path_fn)

        if isinstance(edge, (tuple, list)):
            edge_values = list(edge)
        elif anchor_node is not None:
            edge_values = [anchor_node, edge]
        else:
            raise TypeError("edge must be an Edge or an endpoint sequence")

        if len(edge_values) == 0:
            raise ValueError("edge endpoint sequence cannot be empty")

        if len(edge_values) == 1:
            if anchor_node is None:
                raise ValueError("single-endpoint edge requires an anchor node")
            # Single endpoint shorthand means "anchor_node -> endpoint".
            frm = anchor_node
            to = self._ensure_node(edge_values[0])
            return Edge(frm=frm, to=to)

        frm = self._ensure_node(edge_values[0])
        to = self._ensure_node(edge_values[1])
        label = ""
        metadata = None

        if len(edge_values) >= 3 and edge_values[2] is not None:
            label = str(edge_values[2])
        # Fourth slot is reserved for EdgeMeta; other extras are intentionally ignored.
        if len(edge_values) >= 4 and isinstance(edge_values[3], EdgeMeta):
            metadata = edge_values[3]

        if metadata is None:
            return Edge(frm=frm, to=to, label=label)
        return Edge(frm=frm, to=to, label=label, metadata=metadata)

    @staticmethod
    def _iter_edge_inputs(edges):
        """Normalize `add_node(..., edges=...)` into a list of edge-like values."""
        if isinstance(edges, tuple):
            return [edges]
        if isinstance(edges, (list, set, frozenset)):
            return list(edges)
        return [edges]

    @staticmethod
    def _normalize_attached_edge_input(edge_input, *, anchor_node):
        """Normalize one attached-edge item into an `add_edge`-compatible value."""
        if edge_input is None:
            return None
        if isinstance(edge_input, Edge):
            return edge_input
        if isinstance(edge_input, (tuple, list)):
            if len(edge_input) == 0:
                return None
            if len(edge_input) == 1:
                # Single endpoint shorthand means "anchor_node -> endpoint".
                return (anchor_node, edge_input[0])
            return edge_input
        return (anchor_node, edge_input)

    def add_node(self, node, edges=None):
        """Insert/reuse a node and optionally coerce/add attached edge shorthand definitions."""
        node_obj = self._ensure_node(node)

        if edges is None:
            return node_obj

        for edge_input in self._iter_edge_inputs(edges):
            normalized_edge = self._normalize_attached_edge_input(edge_input, anchor_node=node_obj)
            if normalized_edge is None:
                continue
            self.add_edge(normalized_edge)

        return node_obj

    def add_edge(self, edge):
        """Insert an edge after coercion and return the canonical `Edge` object."""
        edge_obj = self._coerce_edge(edge)
        self.edges.append(edge_obj)
        return edge_obj

    def wiggle(
        self,
        speed: float,
        duration: float,
        temperature: float,
        *,
        windup: float = 0.7,
        winddown: float = 0.7,
    ):
        """Start synchronized wiggle animation for all nodes; returns `self`."""
        speed_value = _finite_float(speed, name="speed")
        if speed_value <= 0.0:
            raise ValueError("speed must be > 0")
        duration_value = _normalize_duration(duration)
        temperature_value = _finite_float(temperature, name="temperature")
        if temperature_value < 0.0 or temperature_value > 1.0:
            raise ValueError("temperature must be between 0 and 1 inclusive")

        now = time.perf_counter()
        for node in self.nodes:
            node._start_wiggle(
                speed=speed_value,
                duration=duration_value,
                temperature=temperature_value,
                windup=windup,
                winddown=winddown,
                now=now,
            )
        return self

    def __str__(self):
        """Return a readable multiline summary of nodes and directed edges."""
        node_descriptions = ", ".join(
            node.id if node.label == node.id else f"{node.id} ({node.label})"
            for node in self.nodes
        ) or "(none)"
        edge_lines = [f"  - {edge.frm.id} -> {edge.to.id}" for edge in self.edges]
        edges_block = "\n".join(edge_lines) if edge_lines else "  - (none)"
        return f"Graph:\n  Nodes: {node_descriptions}\n  Edges:\n{edges_block}"

    def __repr__(self):
        """Mirror `__str__` for debugger-friendly display."""
        return self.__str__()


## Graph Setup
Graph `graph_1` includes default and all available built-in edge styles.

In [ ]:
graph_1 = Graph(
    nodes=["a", "b", "c", "d", ("e", "$ e_1 $")],
    edges=[("a", "b"), ("b", "c"), ("c", "d"), ("d", "e"), ("e", "a"),
           ("a", "c"), ("a", "d")]
)


In [ ]:
graph_1 = Graph(
    nodes=[
        "a",
        ("b", r"$ v_1 $"),
        ("c", r"$ v_2 $"),
        "d",
        ("e", r"$ \sin(e^x) $"),
    ],
    edges=[
        ("a", "b", "default"),
        ("b", "c", "bezier", EdgeMeta(path_style="bezier", path_params={"curvature": 0.30})),
        ("c", "d", "orthogonal", EdgeMeta(path_style="orthogonal", path_params={"orientation": "vertical", "bend": 16})),
        ("d", "e", "polyline", EdgeMeta(path_style="polyline", path_params={"points": [{"x": 250, "y": -170}, {"x": 320, "y": 30}]})),
        ("e", "a", "arc", EdgeMeta(path_style="arc", path_params={"curvature": 0.45})),
        ("a", "c", "straight", EdgeMeta(path_style="straight")),
        Edge(
            frm=Node("b"),
            to=Node("e"),
            label="path_fn",
            path_fn=lambda edge: {"style": "bezier", "params": {"curvature": -0.25}},
        ),
    ],
    metadata=GraphMeta(label="graph_1 edge style gallery", display_label=False),
)

graph_1


## Highlight Example
Demonstrates temporary node/edge emphasis with smooth return to baseline.

In [ ]:
highlight_node = next(node for node in graph_1.nodes if node.id == "b")
highlight_edge = next(edge for edge in graph_1.edges if edge.frm.id == "b" and edge.to.id == "c")

highlight_node.highlight(duration=4.5, scale=1.25, color="#dc2626", pulse=True)
highlight_edge.highlight(duration=4.5, scale=1.8, color="#dc2626", pulse=True)
graph_1


## Animation Cells
Each cell below triggers one distinct animation method.

In [ ]:
graph_1.wiggle(speed=1.2, duration=8.0, temperature=0.45)
graph_1


In [ ]:
next(edge for edge in graph_1.edges if edge.frm.id == "b" and edge.to.id == "c").change_width(to=10, duration=3.0)
graph_1


In [ ]:
next(edge for edge in graph_1.edges if edge.frm.id == "e" and edge.to.id == "a").change_color(to="#dc2626", duration=3.0)
graph_1


In [ ]:
next(node for node in graph_1.nodes if node.id == "a").move_to(x=-180, y=130, duration=3.5)
graph_1


In [ ]:
next(node for node in graph_1.nodes if node.id == "b").change_diameter(to=78, duration=3.0)
graph_1


In [ ]:
next(node for node in graph_1.nodes if node.id == "c").change_fill_color(to="#14b8a6", duration=3.0)
graph_1


In [ ]:
next(node for node in graph_1.nodes if node.id == "d").change_stroke_color(to="#f97316", duration=3.0)
graph_1


In [ ]:
next(node for node in graph_1.nodes if node.id == "e").change_stroke_width(to=9, duration=3.0)
graph_1


In [ ]:
next(node for node in graph_1.nodes if node.id == "b").change_label_size(to=42, duration=3.0)
graph_1


In [ ]:
next(node for node in graph_1.nodes if node.id == "a").scale(factor=1.35, duration=3.0)
graph_1
